# Laboratorio 10 — Reducción de Dimensionalidad
**Curso:** Minería de Datos (EIN132A25)

## Objetivos
- Entender la **maldición de la dimensionalidad**
- Aplicar **PCA** y analizar la varianza explicada
- Visualizar datos de alta dimensión en 2D
- Conocer **t-SNE** como alternativa para visualización

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE

digits = load_digits()
X = digits.data
y = digits.target
print(f"Shape de X: {X.shape}")
print(f"Clases: {np.unique(y)}")

## 1. Visualizar algunas imágenes

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, img, label in zip(axes.flatten(), X[:10], y[:10]):
    ax.imshow(img.reshape(8, 8), cmap="gray")
    ax.set_title(f"Dígito: {label}")
    ax.axis("off")
plt.suptitle("Muestras del dataset de dígitos")
plt.tight_layout()
plt.show()

## 2. Escalar y analizar varianza explicada

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA()
pca_full.fit(X_scaled)
varianza_acum = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(varianza_acum) + 1), varianza_acum, marker="o", markersize=3)
plt.axhline(y=0.90, color="red", linestyle="--", label="90% varianza")
plt.axhline(y=0.95, color="orange", linestyle="--", label="95% varianza")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza explicada acumulada")
plt.title("PCA — Varianza explicada acumulada")
plt.legend()
plt.grid(True)
plt.show()

n_95 = np.argmax(varianza_acum >= 0.95) + 1
print(f"Componentes para 95% de varianza: {n_95} (de 64 originales)")

## 3. Reducir a 2 componentes (visualización)

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)
print(f"Shape original: {X_scaled.shape}")
print(f"Shape reducido: {X_2d.shape}")
print(f"Varianza explicada: {pca_2d.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap="tab10", alpha=0.7, s=20)
plt.colorbar(scatter, label="Dígito")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} var)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} var)")
plt.title("Dígitos proyectados en 2 componentes principales")
plt.show()

## 4. Reducir para modelado (95% varianza)

In [ ]:
pca_95 = PCA(n_components=0.95)
X_reducido = pca_95.fit_transform(X_scaled)
print(f"Shape original: {X_scaled.shape}")
print(f"Shape reducido (95% var): {X_reducido.shape}")

## 5. Reconstrucción desde 2 componentes

In [ ]:
X_reconstruido = pca_2d.inverse_transform(X_2d)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, (ax_orig, ax_rec) in enumerate(zip(axes[0], axes[1])):
    ax_orig.imshow(X[i].reshape(8, 8), cmap="gray")
    ax_orig.set_title("Original")
    ax_orig.axis("off")
    ax_rec.imshow(X_reconstruido[i].reshape(8, 8), cmap="gray")
    ax_rec.set_title("2 PCs")
    ax_rec.axis("off")
plt.suptitle("Original vs reconstruido con 2 componentes principales")
plt.tight_layout()
plt.show()

## 6. t-SNE para visualización

In [ ]:
pca_50 = PCA(n_components=50)
X_50 = pca_50.fit_transform(X_scaled)

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_50)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap="tab10", alpha=0.8, s=20)
plt.colorbar(scatter, label="Dígito")
plt.title("Dígitos — t-SNE (perplexity=30)")
plt.axis("off")
plt.show()

## Ejercicios

### Ejercicio 1 — PCA sobre Titanic

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["Cabin"])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
embarked_dummies = pd.get_dummies(df["Embarked"], prefix="Embarked", drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked_Q", "Embarked_S"]
X_t = StandardScaler().fit_transform(df[features])

pca_t = PCA()
pca_t.fit(X_t)
varianza_t = np.cumsum(pca_t.explained_variance_ratio_)
n_90 = np.argmax(varianza_t >= 0.90) + 1
print(f"Componentes para 90% varianza en Titanic: {n_90}")

### Ejercicio 2 — PCA vs t-SNE sobre Titanic

In [ ]:
y_t = df["Survived"]

pca_2 = PCA(n_components=2)
X_t_2d = pca_2.fit_transform(X_t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (X_vis, titulo) in zip(axes, [(X_t_2d, "PCA 2D")]):
    scatter = ax.scatter(X_vis[:, 0], X_vis[:, 1], c=y_t, cmap="Set1", alpha=0.6, s=20)
    ax.set_title(f"Titanic — {titulo}")
    plt.colorbar(scatter, ax=ax, label="Survived")
plt.tight_layout()
plt.show()

### Desafío — Pipeline completo con PCA

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, train_test_split

X_digits = digits.data
y_digits = digits.target

pipeline_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("clf", DecisionTreeClassifier(random_state=42))
])

pipeline_sin_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(random_state=42))
])

scores_con = cross_val_score(pipeline_pca, X_digits, y_digits, cv=5, scoring="accuracy")
scores_sin = cross_val_score(pipeline_sin_pca, X_digits, y_digits, cv=5, scoring="accuracy")

print(f"Con PCA    (95% var): {scores_con.mean():.4f} ± {scores_con.std():.4f}")
print(f"Sin PCA:             {scores_sin.mean():.4f} ± {scores_sin.std():.4f}")